# 🌌 MURE-AGI Master Pipeline (v7.3)
**The Official Combined System: Train, Host, & Serve MURE AGI**

This notebook combines all three major MURE systems into one flawless, robust pipeline:
1. **Neural Auto-Trainer** (Qwen 7B/1.5B with Unsloth)
2. **Knowledge Graph Synchronization** (SVO CC Brain Integration)
3. **FastAPI Backend Server** (Ngrok tunneling to React Frontend)

> ** Hardware Requirement:** Please ensure you are using a GPU runtime (`Runtime > Change runtime type > Hardware accelerator > T4 GPU`)

In [ ]:
import torch
import sys
from IPython.display import display, HTML

def verify_hardware():
    if not torch.cuda.is_available():
        display(HTML("<div style='padding:20px;background:#ffe6e6;color:#a30000;border-left:5px solid #a30000;font-size:16px;'><b>🚨 FATAL ERROR: GPU required!</b><br/>Please check 'Runtime -> Change runtime type' and select 'T4 GPU' before running.</div>"))
        raise RuntimeError("FATAL ERROR: GPU required for training. Please change runtime to T4 GPU.")
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ Hardware Active: {gpu_name} ({vram:.1f}GB VRAM)")
    return True

verify_hardware()

In [ ]:
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio pydantic
import os
from google.colab import drive

print("📥 Mounting Google Drive...")
drive.mount('/content/drive')

BRAIN_PATH = '/content/drive/MyDrive/svo cc brain/rules'
os.makedirs(BRAIN_PATH, exist_ok=True)
print(f"✅ Knowledge Graph Path configured: {BRAIN_PATH}")

In [ ]:
import json, unicodedata, re

class MURE:
    def __init__(self, path):
        self.path = path
        self.rules = []
        self.index = {}
        if os.path.exists(path):
            try:
                with open(path, 'r', encoding='utf-8') as f:
                   data = json.load(f)
                   # Handle both array format and object format with 'causalMemory'
                   self.rules = data.get('causalMemory', []) if isinstance(data, dict) else data
            except Exception as e:
                print(f"⚠️ Error reading rules: {e}")
        self._reindex()
        print(f"🧠 MURE Initialized with {len(self.rules)} rules.")

    def _reindex(self):
        self.index = {}
        for r in self.rules:
            c = r.get('cause', '').lower()
            if c not in self.index: self.index[c] = []
            self.index[c].append(r)

    def reason(self, text):
        text = unicodedata.normalize('NFC', text).lower()
        res = {'cause': text, 'effect': None, 'strength': 0, 'found': False}
        # Simplified exact and partial matching for safety
        for cause, matches in self.index.items():
             if cause in text or text in cause:
                 best = max(matches, key=lambda x: x.get('strength', 0))
                 res['effect'] = best['effect']
                 res['strength'] = best['strength']
                 res['found'] = True
                 break
        return res

mure_soul = MURE(f'{BRAIN_PATH}/rules.json')

In [ ]:
# API SERVER (NGROK TUNNEL)
# To connect React UI with this Colab Notebook.
NGROK_TOKEN = "YOUR_NGROK_TOKEN" # <-- Put your ngrok token here

import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title='MURE Master API Backend')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class ChatRequest(BaseModel):
    message: str
    settings: dict = None

@app.get('/health')
def health_check():
    return {'status': 'online', 'version': 'MURE Master AGI Pipeline'}

@app.get('/stats')
def get_stats():
    return {'causalRules': len(mure_soul.rules)}

@app.post('/chat')
def chat(req: ChatRequest):
    try:
        # Consensus Logic Execution Route
        logic = mure_soul.reason(req.message)
        
        # Simplified Generation Response (Plug in LLM model here later!)
        if logic['found']:
            reply = f"Based on neural logic: {logic['cause']} causes {logic['effect']}."
        else:
            reply = "Neural extension could not find a match. Attempting fallback..."

        return {
            'reply': reply,
            'frame': {'effect': logic['effect'] if logic['found'] else None},
            'learned': False,
            'source': 'mure_master_colab_api',
            'stats': {'causalRules': len(mure_soul.rules)}
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    print("⚠️ PLEASE INSERT YOUR NGROK TOKEN TO START API. Skipping API launch...")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print('==========================================================')
    print(f'⭐ CONNECT REACT TO THIS URL: {public_url.public_url} ⭐')
    print('==========================================================')
    nest_asyncio.apply()
    uvicorn.run(app, host='0.0.0.0', port=8000)
